In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Use matplotlib default font family with bold text/labels/borders
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 20,
    "font.weight": "bold",
    "axes.linewidth": 2.5,
    "axes.labelsize": 20,
    "axes.labelweight": "bold",
    "axes.titlesize": 20,
    "axes.titleweight": "bold",
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "xtick.major.size": 5,
    "ytick.major.size": 5,
    "xtick.major.width": 2.0,
    "ytick.major.width": 2.0,
    "xtick.minor.size": 2.5,
    "ytick.minor.size": 2.5,
    "xtick.minor.width": 1.5,
    "ytick.minor.width": 1.5,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "lines.linewidth": 2,
    "lines.markersize": 6,
    "legend.frameon": False,
    "legend.fontsize": 14,
    "legend.loc": "best",
    "figure.figsize": (8, 6),
})

def style_axes_bold(ax, spine_width=2.5):
    """Apply bold styling to axis border, labels, and tick labels."""
    for spine in ax.spines.values():
        spine.set_color("black")
        spine.set_linewidth(spine_width)
    ax.tick_params(axis="both", which="major", width=2.0)
    ax.tick_params(axis="both", which="minor", width=1.5)
    ax.xaxis.label.set_fontweight("bold")
    ax.yaxis.label.set_fontweight("bold")
    for tick_label in ax.get_xticklabels() + ax.get_yticklabels():
        tick_label.set_fontweight("bold")

data_color = '#1f77b4'
fit_color = '#d62728'
marker_style = 'o'

In [ ]:
# load the data
data=[]
folder_path=["data/250412-2_result.csv","data/250412-3_result.csv",
             "data/250412-A2_result.csv","data/250412-A4_result.csv",
             "data/250412-A5_result.csv","data/250412-A7_result.csv",
             "data/250412-A9_result.csv","data/250412-4_result.csv",
             "data/250414-4_result.csv"]
for i in folder_path:
    data.append(pd.read_csv(i,index_col=0))
data = pd.concat(data, axis=0)
data.head()

In [ ]:
def log_func(x, a, b):
    return a * np.log(x) + b

In [ ]:
def first_derivative(x,a):
    return a/x

In [ ]:
x = {"BTB count": data["BTB_count"].values,
     "SBU count": data["SBU_count"].values}
y = data["Total Entropy (S)"].values
for key, value in x.items():
    params, covariance = curve_fit(log_func, value, y)
    a, b = params  

    x_fit = np.linspace(min(value), max(value), 200)
    y_fit = log_func(x_fit, a, b)
    dy = first_derivative(x_fit, a)
    base_name = key.lower().replace(" ", "_")

    # R²
    y_pred = log_func(value, a, b)
    r2 = r2_score(y, y_pred)

    plt.figure()
    plt.scatter(value, y, 
                color=data_color, 
                marker=marker_style, 
                s=20, label="Data")
    plt.plot(x_fit, y_fit, 
             color=fit_color, 
             linewidth=2.5, 
             label=f"Fit: y = {a:.3f}·ln(x) + {b:.3f}\nR² = {r2:.3f}")
    plt.xlabel(key, weight='bold')
    plt.ylabel("Total Entropy (S/R)", weight='bold')
    plt.gca().xaxis.set_major_locator(ticker.MaxNLocator(nbins=5)) 
    style_axes_bold(plt.gca())
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{base_name}_entropy_fit_transparent.png", dpi=300, bbox_inches="tight", transparent=True)
    plt.show()

    plt.figure()
    plt.plot(x_fit, dy, 
             color=fit_color, 
             linewidth=2.0, 
             label=f"Derivative: {a:.3f}/x")
    plt.xlabel(key, weight='bold')
    plt.ylabel("dS/dx(R)", weight='bold')
    plt.gca().xaxis.set_major_locator(ticker.MaxNLocator(nbins=5)) 
    style_axes_bold(plt.gca())
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{base_name}_entropy_derivative_transparent.png", dpi=300, bbox_inches="tight", transparent=True)
    plt.show()